# Constraint-based modelling in practice
**MSc Microbiome in Health and Disease — Systems biology workshop (3 hours)**

In the lecture this morning you met the constraint-based toolbox. In this workshop you use each tool on real models:

| Part | Question | Method | Time |
|---|---|---|---|
| A | What can *E. coli* grow on? | FBA with different media | 25 min |
| B | Which nutrient limits growth? | phenotypic phase plane | 25 min |
| C | Growth or product? | production envelope, other objectives | 25 min |
| — | break | | 15 min |
| D | One optimum, many solutions | pFBA, FVA, loopless FVA, sampling | 30 min |
| E | What happens when genes are deleted? | synthetic lethality, FBA vs MOMA | 25 min |
| F | What happens over time? | dynamic FBA | 25 min |

Cells marked **TODO** are for you to complete. Questions marked **Q** are for your report: write short answers
(numbers + one or two sentences of explanation) as you go.

We use two models: the **E. coli core model** (`textbook`: 95 reactions, fast) for most tasks, and the
genome-scale **iJO1366** (2,583 reactions) where scale matters. Both are downloaded by cobrapy the first time
(internet needed).

In [ ]:
import cobra
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")      # hide solver warnings about infeasible test cases

print("cobra", cobra.__version__)
core = cobra.io.load_model("textbook")   # E. coli core model
print(len(core.reactions), "reactions,", len(core.metabolites), "metabolites,", len(core.genes), "genes")

## Part A — The medium is a constraint (25 min)
*Last week (Frederick's notebook) you compared aerobic and anaerobic growth on glucose and blocked uptake reactions
one at a time. Here we go further: many carbon sources, per-carbon yields, and the genome-scale model.*

The **medium** of a model is the set of exchange reactions that are allowed to take something up, and how fast.
In cobrapy, `model.medium` is a dictionary `{exchange reaction: maximal uptake rate}`.

In [ ]:
print(core.medium)
print("growth on the default medium:", round(core.slim_optimize(), 3), "per hour")

**A1.** The function below replaces glucose by another carbon source (same uptake rate, 10 mmol/gDW/h) and
optionally removes oxygen. Note the `with core:` block: every change inside it is undone afterwards, so the model
stays unchanged.

In [ ]:
def growth_on(model, carbon_ex, uptake=10.0, aerobic=True):
    """Maximal growth with `carbon_ex` as the only carbon source."""
    with model:
        medium = model.medium
        medium["EX_glc__D_e"] = 0.0          # remove glucose
        medium[carbon_ex] = uptake           # add the new carbon source
        if not aerobic:
            medium["EX_o2_e"] = 0.0          # remove oxygen
        model.medium = medium
        value = model.slim_optimize(error_value=0.0)
    return round(value, 4)

print(growth_on(core, "EX_glc__D_e"), growth_on(core, "EX_glc__D_e", aerobic=False))

In [ ]:
# TODO: complete this cell
carbons = {"EX_glc__D_e": 6, "EX_fru_e": 6, "EX_pyr_e": 3, "EX_lac__D_e": 3, "EX_succ_e": 4,
           "EX_mal__L_e": 4, "EX_fum_e": 4, "EX_akg_e": 5, "EX_glu__L_e": 5, "EX_ac_e": 2, "EX_etoh_e": 2}
rows = []
for ex, n_carbon in carbons.items():
    # TODO: compute aerobic and anaerobic growth with growth_on(), and the aerobic growth per C-mmol
    aer, ana = ..., ...
    rows.append({"carbon source": core.reactions.get_by_id(ex).reactants[0].name, "C atoms": n_carbon,
                 "aerobic": aer, "anaerobic": ana, "aerobic growth per C-mmol": ...})
table_A = pd.DataFrame(rows).sort_values("aerobic", ascending=False)
table_A

**Q A1.** Which carbon sources support growth without oxygen, and which do not? Suggest why (think about how
ATP and NAD⁺ are regenerated without respiration).

**Q A2.** Glucose gives the highest growth per mmol. Is it also the best *per carbon atom*? What does that tell you
about comparing growth rates on different substrates?

**A2 (genome scale).** Now repeat a few carbon sources in the genome-scale model iJO1366. The first load downloads
the model (a few seconds).

In [ ]:
ijo = cobra.io.load_model("iJO1366")
print(len(ijo.reactions), "reactions,", len(ijo.genes), "genes")
for ex in ["EX_glc__D_e", "EX_glyc_e", "EX_succ_e", "EX_ac_e", "EX_xyl__D_e"]:
    print(f"{ex:14s} aerobic {growth_on(ijo, ex):.3f}   anaerobic {growth_on(ijo, ex, aerobic=False):.3f}")

**Q A3.** iJO1366 can grow on xylose and glycerol; the core model has no reactions for them. Which model
would you use to predict growth on a new substrate, and why?

## Part B — Which nutrient limits growth? A phenotypic phase plane (25 min)
We vary two uptake rates, glucose and oxygen, and compute the maximal growth for every combination (about 300 FBAs,
a few seconds). We also record acetate secretion.

In [ ]:
# TODO: complete this cell
glc_values = np.arange(0, 21, 1.0)
o2_values = np.arange(0, 31, 2.0)
growth = np.zeros((len(o2_values), len(glc_values)))
acetate = np.zeros_like(growth)
for i, o2 in enumerate(o2_values):
    for j, glc in enumerate(glc_values):
        with core:
            # TODO: set the lower bounds of EX_glc__D_e and EX_o2_e (uptake = negative flux), then optimise
            # and store the objective value in growth[i, j] and the EX_ac_e flux in acetate[i, j]
            ...

fig = go.Figure(go.Contour(z=growth, x=glc_values, y=o2_values, colorscale="Viridis",
                           colorbar=dict(title="growth (1/h)")))
fig.update_layout(xaxis_title="glucose uptake (mmol/gDW/h)", yaxis_title="oxygen uptake (mmol/gDW/h)",
                  title="Phenotypic phase plane: maximal growth", width=650, height=500)
fig.show()

In [ ]:
fig = go.Figure(go.Contour(z=acetate, x=glc_values, y=o2_values, colorscale="Oranges",
                           colorbar=dict(title="acetate (mmol/gDW/h)")))
fig.update_layout(xaxis_title="glucose uptake", yaxis_title="oxygen uptake",
                  title="Acetate secretion at maximal growth", width=650, height=500)
fig.show()

# the 'line of optimality': for each glucose uptake, the smallest oxygen uptake that gives (almost) maximal growth
line = [o2_values[np.argmax(growth[:, j] >= growth[:, j].max() - 1e-6)] for j in range(len(glc_values))]
print(pd.DataFrame({"glucose": glc_values, "O2 on line of optimality": line}).iloc[::4].to_string(index=False))

**Q B1.** At a glucose uptake of 10, above which oxygen uptake does growth stop increasing? What is the
O₂ : glucose ratio on the line of optimality, and how does it compare with the 6 : 1 of complete respiration?

**Q B2.** Where in the plane is acetate secreted? Explain this 'overflow metabolism' using Tuesday's
respire-or-ferment exercise.

**Q B3 (shadow prices).** Pick one point above and one below the line of optimality (e.g. glucose 10 with O₂ 30,
and glucose 10 with O₂ 10). Run FBA at both and look at `sol.shadow_prices[["glc__D_e", "o2_e"]]`. Which nutrient has
a non-zero shadow price in each region, and what does that mean?

In [ ]:
for o2 in (30, 10):
    with core:
        core.reactions.EX_glc__D_e.lower_bound = -10
        core.reactions.EX_o2_e.lower_bound = -o2
        sol = core.optimize()
        print(f"O2 uptake ≤ {o2}: growth {sol.objective_value:.3f}; shadow prices:",
              sol.shadow_prices[["glc__D_e", "o2_e"]].round(4).to_dict())

## Part C — Growth or product? Objectives and trade-offs (25 min)
Without oxygen, *E. coli* ferments. Which products can it make, and does making them cost growth? A **production
envelope** shows, for each growth rate, the minimal and maximal possible secretion of a product.

In [ ]:
from cobra.flux_analysis import production_envelope

anaerobic = core.copy()
anaerobic.reactions.EX_o2_e.lower_bound = 0
print("anaerobic maximal growth:", round(anaerobic.slim_optimize(), 3))

env = production_envelope(anaerobic, ["Biomass_Ecoli_core"], objective="EX_etoh_e", points=20)
fig = go.Figure()
fig.add_trace(go.Scatter(x=env["Biomass_Ecoli_core"], y=env["flux_maximum"], name="maximum", mode="lines"))
fig.add_trace(go.Scatter(x=env["Biomass_Ecoli_core"], y=env["flux_minimum"], name="minimum", mode="lines", fill="tonexty"))
fig.update_layout(xaxis_title="growth (1/h)", yaxis_title="ethanol secretion (mmol/gDW/h)",
                  title="Production envelope: ethanol (anaerobic)", width=650, height=450)
fig.show()

In [ ]:
# TODO: complete this cell
products = {"ethanol": "EX_etoh_e", "acetate": "EX_ac_e", "formate": "EX_for_e",
            "lactate": "EX_lac__D_e", "succinate": "EX_succ_e"}
mu_max = anaerobic.slim_optimize()
rows = []
for name, ex in products.items():
    with anaerobic:
        anaerobic.objective = ex                       # maximise the product instead of growth
        max_at_zero_growth = anaerobic.slim_optimize()
    with anaerobic:
        # TODO: force growth to (almost) its maximum by setting the lower bound of Biomass_Ecoli_core,
        # then minimise and maximise the product (objective_direction = "min" / "max")
        min_at_max_growth = ...
        max_at_max_growth = ...
    rows.append({"product": name, "max (any growth)": round(max_at_zero_growth, 2),
                 "min at max growth": min_at_max_growth, "max at max growth": max_at_max_growth})
table_C = pd.DataFrame(rows)
table_C

**Q C1.** What is the maximal ethanol secretion without growth, and at maximal growth? Explain the
trade-off in terms of where the carbon and the ATP go.

**Q C2.** A product is *growth-coupled* if the cell must make it in order to grow at its maximum (minimum > 0 at
maximal growth). Which products are growth-coupled anaerobically? Why is growth coupling attractive for metabolic
engineering (think of OptKnock)?

**Q C3.** Last week you maximised ATP maintenance (`ATPM`) with oxygen (175 mmol from 10 glucose). Now change the objective to ATP maintenance (`ATPM`, set its upper bound to 1000) in the anaerobic model.
How much ATP can 10 glucose give without oxygen, and why is it more than 2 ATP per glucose?

In [ ]:
with anaerobic:
    anaerobic.objective = "ATPM"
    anaerobic.reactions.ATPM.upper_bound = 1000
    sol = anaerobic.optimize()
    print("max ATP anaerobic:", round(sol.objective_value, 2))
    print(sol.fluxes[[r.id for r in anaerobic.exchanges]].loc[lambda s: s.abs() > 1e-6].round(2))

## Part D — One optimum, many solutions (30 min)
FBA returns **one** flux distribution, but many may reach the same growth. Here we compare four views of the
solution space: FBA, pFBA, FVA (with and without loops) and sampling.

In [ ]:
from cobra.flux_analysis import pfba, flux_variability_analysis

fba_sol = core.optimize()
pfba_sol = pfba(core)
print("growth FBA", round(fba_sol.objective_value, 4), "| growth pFBA", round(pfba_sol.fluxes["Biomass_Ecoli_core"], 4))
print("total flux FBA", round(fba_sol.fluxes.abs().sum(), 1), "| total flux pFBA", round(pfba_sol.fluxes.abs().sum(), 1))

In [ ]:
# TODO: complete this cell
fva_100 = flux_variability_analysis(core, fraction_of_optimum=1.0)
fva_90 = ...   # TODO: FVA at 90% of the optimum
variable_100 = fva_100[(fva_100.maximum - fva_100.minimum) > 1e-6]
variable_90 = ...   # TODO
print("reactions that can vary at 100% of max growth:", len(variable_100))
print("reactions that can vary at 90% of max growth:", len(variable_90))
variable_100

In [ ]:
loopless = flux_variability_analysis(core, ["FRD7", "SUCDi"], fraction_of_optimum=1.0, loopless=True)
pd.concat({"ordinary FVA": fva_100.loc[["FRD7", "SUCDi"]], "loopless FVA": loopless}, axis=1).round(2)

**Q D1.** How many reactions can vary at the optimum, and at 90% of it? What does this say about how
precisely FBA determines individual fluxes?

**Q D2.** FRD7 and SUCDi have huge ranges in ordinary FVA. Look up both reactions (`core.reactions.FRD7`). Why can
they carry any flux without changing anything else, and why does loopless FVA remove this?

**Sampling.** Draw 1,000 random flux distributions with growth at least 90% of its maximum (about a minute).

In [ ]:
from cobra.sampling import sample

with core:
    core.reactions.Biomass_Ecoli_core.lower_bound = 0.9 * core.slim_optimize()
    samples = sample(core, 1000, method="optgp", seed=42, processes=1)

reactions = ["PGI", "G6PDH2r", "PFK", "CS"]
long = samples[reactions].melt(var_name="reaction", value_name="flux")
fig = px.histogram(long, x="flux", facet_col="reaction", nbins=40, facet_col_wrap=4, height=350)
fig.update_xaxes(matches=None)
fig.show()
summary = pd.DataFrame({"FBA": fba_sol.fluxes[reactions], "pFBA": pfba_sol.fluxes[reactions],
                        "FVA90 min": fva_90.loc[reactions, "minimum"], "FVA90 max": fva_90.loc[reactions, "maximum"],
                        "sample mean": samples[reactions].mean()}).round(2)
summary

**Q D3.** For PGI (glycolysis) and G6PDH2r (pentose phosphate pathway), compare the FBA value, the FVA range
and the sample distribution. Which one would you report if someone asked "how much flux goes through the pentose
phosphate pathway?", and why?

## Part E — What happens when genes are deleted? (25 min)
*Last week you ran single gene deletions. Today: pairs of genes (synthetic lethality, from Tuesday's lecture) and
how a mutant responds (FBA versus MOMA, from this morning).*

In [ ]:
# TODO: complete this cell
from cobra.flux_analysis import single_gene_deletion, double_gene_deletion

wt_growth = core.slim_optimize()
single = single_gene_deletion(core)
single["gene"] = [list(ids)[0] for ids in single["ids"]]
essential = set(single[(single.growth.isna()) | (single.growth < 0.01 * wt_growth)]["gene"])
non_essential = [g.id for g in core.genes if g.id not in essential]
print(len(essential), "essential genes;", len(non_essential), "non-essential genes")

# TODO: run double_gene_deletion with gene_list1=["b4025"] (pgi) and gene_list2=non_essential,
# and keep the pairs whose growth is below 1% of wt_growth (or NaN): these are pgi's synthetic lethal partners
pairs = ...
lethal = ...
lethal

**Q E1.** List the synthetic lethal partners of *pgi*. Group them by reason: which are alternative routes for
glucose-6-phosphate, and which are not obviously related? (Challenge: without PGI, all glucose goes through the
oxidative pentose phosphate pathway, which makes NADPH. Why might complex I (*nuo* genes) then become essential?
Compare the fluxes of NADTRHD (transhydrogenase) and NADH16 in the wild type and the *pgi* mutant, e.g. with pfba().)

**FBA versus MOMA.** FBA assumes a mutant immediately reaches its new optimum; MOMA assumes it stays as close as
possible to the wild-type fluxes. We compare the growth both predict for five knockouts in central metabolism.
(We use the linear version of MOMA, which works with the free GLPK solver.)

In [ ]:
# TODO: complete this cell
from cobra.flux_analysis import moma

knockouts = {"pgi": "b4025", "zwf": "b1852", "tpiA": "b3919", "gltA": "b0720", "sdhA": "b0723"}
wild_type = core.optimize()          # the reference flux distribution MOMA stays close to
rows = []
for name, gene in knockouts.items():
    with core:
        core.genes.get_by_id(gene).knock_out()
        # TODO: compute the FBA growth of the mutant (slim_optimize), and the MOMA growth with
        # moma(core, solution=wild_type, linear=True); read the "Biomass_Ecoli_core" flux of the MOMA solution
        # (Why must the reference be the WILD-TYPE solution, computed before the knockout?)
        fba_growth = ...
        moma_growth = ...
    rows.append({"knockout": f"{name} ({gene})", "FBA growth": fba_growth, "MOMA growth (linear)": moma_growth})
table_E = pd.DataFrame(rows)
table_E

**Q E2.** For which knockouts do FBA and MOMA disagree most? Which prediction would you expect to match a
freshly made mutant, and which one the same mutant after weeks of adaptive laboratory evolution?

**Q E3 (optional, ~2 min).** Run `double_gene_deletion(core, gene_list1=non_essential)` for all pairs. How many
synthetic lethal pairs are there? Pick one pair not discussed in the lecture and explain it from the reactions.

## Part F — What happens over time? Dynamic FBA (25 min)
In a batch culture the medium changes: glucose is used up and products accumulate. Dynamic FBA (static optimisation
approach) repeats three steps every Δt:
1. set uptake bounds from the current concentrations (Michaelis–Menten uptake kinetics),
2. solve FBA,
3. update biomass and concentrations with the predicted growth and exchange fluxes.

In [ ]:
def dynamic_fba(model, o2_max=8.0, X0=0.01, glc0=20.0, ac0=0.0, dt=0.1, t_end=14.0,
                vmax=10.0, km=0.5):
    X, glc, ac = X0, glc0, ac0
    rows = []
    for k in range(int(t_end / dt) + 1):
        with model:
            model.reactions.EX_glc__D_e.lower_bound = -vmax * glc / (km + glc)
            model.reactions.EX_ac_e.lower_bound = -vmax * ac / (km + ac)
            model.reactions.EX_o2_e.lower_bound = -o2_max
            sol = model.optimize()
            if sol.status == "optimal":
                mu, v_glc, v_ac = sol.objective_value, sol.fluxes["EX_glc__D_e"], sol.fluxes["EX_ac_e"]
            else:
                mu, v_glc, v_ac = 0.0, 0.0, 0.0
        rows.append({"time": k * dt, "biomass": X, "glucose": glc, "acetate": ac, "growth rate": mu})
        X, glc, ac = X + mu * X * dt, max(glc + v_glc * X * dt, 0.0), max(ac + v_ac * X * dt, 0.0)
    return pd.DataFrame(rows)

batch = dynamic_fba(core, o2_max=8.0)
fig = px.line(batch, x="time", y=["glucose", "acetate", "biomass"], title="Dynamic FBA, oxygen uptake ≤ 8")
fig.update_layout(yaxis_title="mM (biomass: gDW/L)", width=700, height=420)
fig.show()

In [ ]:
# TODO: complete this cell
results = []
for o2 in (2, 8, 15, 25):
    b = ...   # TODO: run dynamic_fba with o2_max = o2
    results.append({"O2 max": o2, "max acetate (mM)": round(b.acetate.max(), 2),
                    "glucose gone at (h)": b.time[b.glucose < 0.05].min() if (b.glucose < 0.05).any() else None,
                    "final biomass (gDW/L)": round(b.biomass.iloc[-1], 3)})
table_F = pd.DataFrame(results)
table_F

**Q F1.** Describe the two growth phases in the oxygen-limited batch (O₂ ≤ 8). What is a diauxic shift, and
why does the model use glucose before acetate?

**Q F2.** How do maximal acetate and final biomass change with the oxygen supply? Relate this to your phase plane
from Part B.

**Q F3.** Name two things this dynamic FBA leaves out that a real batch culture has (hint: regulation, lag phases,
oxygen transfer).

## Report questions
Write a short report (about 4 pages, figures included; code in an appendix). Use your answers above.

1. **Constraints.** Using Parts A and B, explain how the medium (carbon source, oxygen) constrains the predicted
   phenotype. Include your carbon-source table and the phase plane, and identify which nutrient limits growth in
   each region.
2. **Objective functions.** Using Part C, compare growth, ATP production and product secretion as objectives.
   When is maximal growth a reasonable assumption, and when is it not?
3. **Uncertainty.** Using Part D, discuss how well individual fluxes are determined by FBA. When would you report
   pFBA, FVA ranges or sampled distributions?
4. **Genetic perturbations.** Using Part E, discuss synthetic lethality (with examples explained from the network)
   and the difference between FBA and MOMA predictions.
5. **Dynamics.** Using Part F, explain what dynamic FBA adds to FBA, and its limitations.

*Integration of omics data into models is covered in the integrative analysis lecture and workshop.*